# 1. The data, and the three decisions that drive everythingWhat this notebook establishes, before any model exists:1. what is actually in each dataset,2. which values are measurements and which are not,3. whether there is structure a recommender could exploit.The logic lives in `../src/dta`; this notebook calls it and explains the output.

In [ ]:
import sys, warningssys.path.insert(0, "../src")warnings.filterwarnings("ignore")import numpy as np, pandas as pdpd.set_option("display.width", 180); pd.set_option("display.max_columns", 50)from dta import data as D, analysis as A, kernels as K, plots as Pfrom dta.config import DATASETS, SCORE_NAMEP.use_style()proc = {ds: D.load_processed(ds) for ds in DATASETS}

## Decision 1: entities are chemistry, not identifiersDAVIS lists 442 target IDs but only **379 distinct amino-acid sequences**. Mutants andphosphorylated forms (`ABL1`, `ABL1p`, `ABL1(T315I)`, ...) are separate IDs carrying thesame sequence in the benchmark files.Splitting on IDs would put the identical protein on both sides of a train/test boundary.Pairs are therefore keyed on `(canonical SMILES, amino-acid sequence)`.

In [ ]:
for ds in DATASETS:    print(f"=== {ds.upper()}"); display(proc[ds]["counts"]); display(proc[ds]["cleaning_log"])

### The duplicates disagree, which is why this mattersIf two "different" targets are literally the same sequence but their measured affinitiesdiffer by orders of magnitude, then keeping them apart was recording noise as signal.

In [ ]:
A.duplicate_conflicts(proc["davis"]["pairs"], top=5)

## Decision 2: the assay ceiling is not a measurementDAVIS stopped at Kd = 10,000 nM. A label of pKd 5.0 means "at most 5.0", not "5.0".These are **censored**, flagged and kept: deleting them would throw away the finding thatthose pairs do not bind, which is most of what the dataset knows.

In [ ]:
for ds in DATASETS:    print(f"=== {ds.upper()}"); display(A.censoring_stats(proc[ds]["pairs"]))

## Decision 3: a missing cell is not a zeroThree quarters of KIBA was never tested. Those cells become no row at all. Filling themwith zero would invent roughly 356,000 confident non-binding claims nobody measured.

In [ ]:
mats = {}for ds in DATASETS:    p = proc[ds]["pairs"]    mats[ds] = D.to_matrix(p, proc[ds]["n_lig"], proc[ds]["n_tgt"])    miss = ~np.isfinite(mats[ds])    print(f"{ds.upper():6s} {mats[ds].shape}  missing {miss.sum():,}/{mats[ds].size:,} = {miss.mean():.1%}")P.heatmaps(mats)P.distributions({ds: proc[ds]["pairs"]["score"].to_numpy() for ds in DATASETS},                {ds: proc[ds]["pairs"]["censored"].to_numpy() for ds in DATASETS})

## Is there anything for a recommender to learn?Split the variance into what a drug's own average explains, what a target's averageexplains, both together, and the remainder. That remainder is drug-by-target**interaction**: the part no bias model can reach, and therefore the ceiling on whatcollaborative filtering can add.

In [ ]:
var = {ds: A.variance_decomposition(proc[ds]["pairs"], proc[ds]["n_lig"], proc[ds]["n_tgt"])       for ds in DATASETS}for ds in DATASETS:    print(f"=== {ds.upper()}"); display(var[ds])

In [ ]:
spec = {}for ds in DATASETS:    M = mats[ds]    block = M if np.isfinite(M).all() else A.densest_block(M)    spec[ds] = A.svd_spectrum(block)    print(f"{ds.upper()}: top 5 components hold {spec[ds].loc[4,'cumulative_energy']:.1%}")P.structure(var, spec)

### Does sequence predict binding?The cold-target task replaces a missing protein's history with its sequence, so thattask only makes sense if sequence-similar proteins really do bind similarly.

In [ ]:
for ds in DATASETS:    ks = K.build_kernels(ds, proc[ds])    r = A.similarity_vs_profile(ks["target_sequence"], mats[ds])    print(f"{ds.upper()}: rho = {r['spearman_rho']:.3f} over {r['n_pairs']:,} target pairs")

Reproduce every Task 1 table and figure:

In [ ]:
from dta import run_task1run_task1.main()